<a href="https://colab.research.google.com/github/1Jaffry1/student-workshop/blob/master/04_YOLO_World_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modern Object Detection IV: YOLO-World and Open-Vocabulary Detection

Vision–language detection with text prompts.


## 0. Workshop introduction

Closed-set detectors (YOLO, Faster R-CNN, RT-DETR in the previous notebooks) can only name classes they were trained on (COCO’s 80 names).

**Open-vocabulary** detection takes a **text prompt** at inference time. If you type `red backpack`, the model tries to find that concept even if it was never a dedicated training class.

This notebook is the most interactive one: you will spend most of the time changing language, not code.


## 1. Learning objectives

- Contrast closed-set vs open-vocabulary detection.
- Explain that image features and text features are matched in a shared space.
- See how prompt wording changes detections, including false positives.


## How this workshop is structured

You will **not** implement neural-network layers from scratch.

The instructor cells already contain working functions for each important stage of the algorithm. Your job is to:

1. Read what each stage does and why it exists.
2. Assemble those stages in the correct order (a short coding task).
3. Change one or two parameters and watch the output change.

The demo cell is only a one-liner (`run_full_pipeline`) so you can see a result after Run all. **Do not copy that function for the assembly exercise** — wire the named stages listed in the student task.

Hands-on coding is intentionally light (~20–25% of the session). Most of the time is for understanding the pipeline.


## 2. Environment setup


In [ ]:
!pip install -q ultralytics==8.3.155 matplotlib opencv-python-headless pillow
!pip install -q git+https://github.com/ultralytics/CLIP.git


In [ ]:
import platform
import sys

print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(props.total_memory / 1024 ** 3, 2), "GB")
else:
    print("GPU: None")
    print("GPU memory: n/a")
    print("\nEnable a GPU: Runtime → Change runtime type → T4 GPU, then Restart session.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## Troubleshooting

| Problem | Fix |
|---|---|
| CUDA is unavailable | `Runtime → Change runtime type → T4 GPU`, then restart and Run all |
| Package import fails | `Runtime → Restart session`, then Run all |
| Checkpoint download fails | Re-run the setup / model-load cell |
| Out of memory | Use the smaller default model, or a smaller image |
| A student cell has `???` | That is expected. Fill it in, or set `RUN_STUDENT_ASSEMBLY = False` to skip it |

Do not spend workshop time debugging package conflicts. Restart and Run all first.


The first `encode_text` / `set_classes` call downloads CLIP weights. That happens once. The setup cell already installs the CLIP package.


## 3. Imports


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

import urllib.error
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches
from PIL import Image

SAMPLE_IMAGES = {
    "bus": "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg",
    "zidane": "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/zidane.jpg",
    "cats": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "living_room": "http://images.cocodataset.org/val2017/000000000139.jpg",
    "street": "http://images.cocodataset.org/val2017/000000037777.jpg",
}


def download_image(url, path="sample.jpg"):
    path = Path(path)
    if path.exists():
        return path
    req = urllib.request.Request(url, headers={"User-Agent": "object-detection-workshop/1.0"})
    try:
        with urllib.request.urlopen(req) as resp:
            path.write_bytes(resp.read())
    except urllib.error.HTTPError as err:
        loc = err.headers.get("Location")
        if err.code in (301, 302, 303, 307, 308) and loc:
            return download_image(loc, path)
        raise
    return path


def load_image(path):
    """Load an RGB uint8 image as a NumPy array (H, W, 3)."""
    return np.array(Image.open(path).convert("RGB"))


def _class_color(cls_id):
    rng = np.random.RandomState(int(cls_id) * 17 + 3)
    return rng.randint(40, 230, size=3) / 255.0


def visualize_detections(
    image,
    boxes,
    scores=None,
    labels=None,
    names=None,
    title=None,
    max_dets=60,
    prompt=None,
):
    """Draw xyxy boxes. `names` maps class id → string."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(image)
    ax.axis("off")
    header = title or ""
    if prompt:
        header = (header + "  |  prompt: " + str(prompt)).strip(" |")
    if header:
        ax.set_title(header, fontsize=12)

    boxes = [] if boxes is None else list(boxes)[:max_dets]
    scores = [None] * len(boxes) if scores is None else list(scores)[:max_dets]
    labels = [None] * len(boxes) if labels is None else list(labels)[:max_dets]

    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = [float(v) for v in box]
        cls_id = 0 if label is None else int(label)
        color = _class_color(cls_id)
        ax.add_patch(
            patches.Rectangle(
                (x1, y1),
                max(x2 - x1, 1.0),
                max(y2 - y1, 1.0),
                linewidth=2,
                edgecolor=color,
                facecolor="none",
            )
        )
        name = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else (str(label) if label is not None else "")
        caption = name if score is None else f"{name} {float(score):.2f}"
        ax.text(
            x1,
            max(y1 - 4, 12),
            caption,
            color="white",
            fontsize=9,
            bbox=dict(facecolor=color, edgecolor="none", pad=2, alpha=0.85),
        )
    fig.tight_layout()
    plt.show()
    return fig


## 4. Load an example image


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

IMAGE_PATH = download_image(SAMPLE_IMAGES["bus"], "bus.jpg")
image = load_image(IMAGE_PATH)
plt.figure(figsize=(8, 6)); plt.imshow(image); plt.axis("off"); plt.title("Input image"); plt.show()


## 5–6. The YOLO-World pipeline

```text
Image  → Image encoder → image features  ↘
                                           Vision–language matching → boxes + prompted classes
Text   → Text encoder  → text features   ↗
```

| Closed-set YOLO | YOLO-World |
|---|---|
| Class list is a fixed 80-way layer | Class list is whatever text you pass |
| Cannot detect “person wearing a hat” as a label | Can try that phrase as a prompt |

We use Ultralytics YOLO-World (`yolov8s-worldv2.pt`), which loads the official YOLO-World idea with a Colab-friendly API. Students do not need MMYOLO.


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

from ultralytics import YOLOWorld

WORLD_WEIGHTS = "yolov8s-worldv2.pt"
world = YOLOWorld(WORLD_WEIGHTS)
world.to(DEVICE)
print("Loaded", WORLD_WEIGHTS)


def prepare_text_prompts(classes):
    """Student-facing vocabulary: ordinary English words or short phrases."""
    prompts = [c.strip() for c in classes if str(c).strip()]
    if not prompts:
        raise ValueError("Provide at least one text prompt.")
    return prompts


def encode_text(prompts):
    """CLIP text encoder. Embeddings become the detector's class weights."""
    # Avoid a known tensor-versioning error when changing prompts after a predict().
    if hasattr(world.model, "clip_model"):
        world.model.clip_model = None
    world.set_classes(list(prompts))
    txt = getattr(world.model, "txt_feats", None)
    print("text prompts:", prompts)
    if txt is not None:
        print("text embedding shape:", tuple(txt.shape))
    return prompts


def encode_image(image):
    """Image encoder lives inside the YOLO-World backbone. We keep the RGB image here
    and run it together with the text embeddings at detection time."""
    return image


@torch.no_grad()
def match_image_and_text_features(image, conf=0.1):
    """Vision–language matching + box prediction (official Ultralytics forward)."""
    results = world.predict(image, conf=conf, verbose=False)
    return results[0]


def filter_predictions(result, conf=0.1):
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return {
            "boxes": np.zeros((0, 4), dtype=np.float32),
            "scores": np.zeros((0,), dtype=np.float32),
            "labels": np.zeros((0,), dtype=np.int64),
            "names": dict(world.names),
        }
    scores = boxes.conf.cpu().numpy()
    keep = scores >= conf
    return {
        "boxes": boxes.xyxy.cpu().numpy()[keep],
        "scores": scores[keep],
        "labels": boxes.cls.cpu().numpy().astype(int)[keep],
        "names": dict(world.names),
    }


def show_open_vocab(image, dets, prompts, title=None):
    visualize_detections(
        image,
        dets["boxes"],
        dets["scores"],
        dets["labels"],
        names=dets["names"],
        title=title,
        prompt=", ".join(prompts),
    )


print("YOLO-World helpers ready.")


## 7. Student assembly

Wire text and image into the matcher. Then change the prompt list.


In [ ]:
# ==========================================
# STUDENT TASK
# ==========================================
# Set True after you replace every ??? with the correct function call.
RUN_STUDENT_ASSEMBLY = False

if RUN_STUDENT_ASSEMBLY:
    classes = ["person", "car", "dog"]  # change these
    prompts = ???          # prepare_text_prompts
    _ = ???                # encode_text
    _ = ???                # encode_image
    result = ???           # match_image_and_text_features
    dets = filter_predictions(result, conf=0.1)
    show_open_vocab(image, dets, prompts, title="Student assembly")
    print("detections:", len(dets["boxes"]))
else:
    print('Skipping student assembly. The instructor demo above already ran the pipeline.')
    print('During the exercise: fill in the TODOs, then set RUN_STUDENT_ASSEMBLY = True.')


## 8–9. Experiments — this is the main activity

Try several vocabularies on the **same** image. Then try a second image if you have time (`SAMPLE_IMAGES["zidane"]` or `"street"`).


In [ ]:
# ==========================================
# STUDENT TASK
# ==========================================

CONF = 0.1  # open-vocab scores are often lower than closed-set YOLO; start low

prompt_sets = [
    ["person", "car", "dog"],
    ["red bus", "person wearing a coat", "backpack"],
    ["traffic sign", "wheel", "license plate"],
    ["dragon", "spaceship"],  # absent objects: expect few or no boxes, or false positives
]

for classes in prompt_sets:
    pass
    # TODO ; complete this part


## 10. Think about it

1. What changes when the detector is no longer restricted to a fixed set of classes?
2. Why might “red bus” work better or worse than “bus”?
3. If you prompt for something that is not in the image, what should you look at besides the boxes (hint: confidence and plausible mix-ups)?


## 11. Optional challenge

Write a prompt for an object that is **not** a COCO class (for example `suitcase wheels` or `advertisement on the bus`). Does the model find it, or a nearby substitute?


## 12. Summary

- YOLO-World aligns **image** and **text** features.
- The prompt *is* the class list.
- Open vocabulary is powerful and also easier to fool.

Next: **Cube R-CNN** — from 2D boxes to **3D cuboids** from a single RGB image.
